In [1]:
from tradepy.data.loader import load_future
from tradepy.config.config import load_config, load_symbols, load_feature_config
from tradepy.data.cleaner import add_trading_date_by_gap
from tradepy.data.resampler import resample_ohlcv, daily_ohlcv_cummulative
from tradepy.features.generate import generate_features
from tradepy.features.quality_check import data_quality_report

In [2]:
symbols = load_symbols()

In [3]:
symbols

['AD',
 'BP',
 'CL',
 'EC',
 'ES',
 'GC',
 'MFXI',
 'NG',
 'NQ',
 'YM',
 'ZB',
 'ZN',
 'ZS']

In [4]:
ASSET = symbols[4]

In [5]:
cfg = load_config()

In [6]:
MINUTES             = cfg['sampling_minutes']        # 240
RETURN_HORIZON_MIN  = cfg['return_horizon_min']      # 2880

In [7]:
df, spec = load_future(ASSET)

In [8]:
df = add_trading_date_by_gap(df)

In [9]:
df

,ticker,per,date,time,open,high,low,close,volume,openint,datetime,session_id,trading_date
0,ES,I,1997-09-10,07:08:00,934.00,934.00,934.00,934.00,1,0,1997-09-10 07:08:00,0,1997-09-10
1,ES,I,1997-09-10,07:15:00,933.75,933.75,933.75,933.75,1,0,1997-09-10 07:15:00,0,1997-09-10
2,ES,I,1997-09-10,08:10:00,933.75,933.75,933.75,933.75,1,0,1997-09-10 08:10:00,0,1997-09-10
3,ES,I,1997-09-10,08:13:00,934.00,934.00,934.00,934.00,1,0,1997-09-10 08:13:00,0,1997-09-10
4,ES,I,1997-09-10,08:20:00,933.75,933.75,933.75,933.75,1,0,1997-09-10 08:20:00,0,1997-09-10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8631144,ES,I,2026-01-19,06:57:00,6917.00,6917.00,6916.75,6916.75,26,0,2026-01-19 06:57:00,5073,2026-01-19
8631145,ES,I,2026-01-19,06:58:00,6917.00,6917.00,6916.75,6917.00,33,0,2026-01-19 06:58:00,5073,2026-01-19
8631146,ES,I,2026-01-19,06:59:00,6917.00,6917.50,6916.75,6916.75,112,0,2026-01-19 06:59:00,5073,2026-01-19
8631147,ES,I,2026-01-19,07:00:00,6916.75,6917.00,6916.25,6916.75,64,0,2026-01-19 07:00:00,5073,2026-01-19


In [10]:
df_resampled = resample_ohlcv(df, period=f"{MINUTES}min")
df_cumulative = daily_ohlcv_cummulative(df_resampled)

In [11]:
config = load_feature_config()

In [12]:
config["global"]["sampling_minutes"] = MINUTES
config["global"]["return_horizon_min"] = RETURN_HORIZON_MIN
config["global"]["tick_size"] = spec["tick_size"]

In [13]:
df_final = generate_features(df_cumulative, config)

In [14]:
df_final

,datetime,open,high,low,close,volume,openint,ticker,per,trading_date,...,fib_dist_786,nearest_fib_level,dist_to_nearest_fib,swing_extension,bars_between_swing_highs,bars_between_swing_lows,swing_high_velocity,swing_low_velocity,bars_between_swings,swing_cycle_ratio
0,1997-09-10 08:00:00,934.00,934.00,933.75,933.75,2,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1997-09-10 12:00:00,933.75,934.25,932.75,933.00,28,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1997-09-10 16:00:00,932.75,933.00,928.75,931.75,414,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1997-09-10 20:00:00,932.00,932.00,924.75,926.25,1373,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1997-09-11 00:00:00,926.00,930.25,914.50,914.75,1290,0.0,ES,240min,1997-09-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42589,2026-01-16 16:00:00,7002.75,7004.25,6981.50,6982.00,228308,0.0,ES,240min,2026-01-16,...,-0.615412,0.236,-0.065412,-0.829412,13.0,7.0,7.230769,6.071429,1.0,3.000000
42590,2026-01-16 20:00:00,6982.00,6995.00,6960.50,6987.75,561713,0.0,ES,240min,2026-01-16,...,-0.382899,0.382,0.021101,-0.596899,4.0,7.0,8.062500,6.071429,3.0,0.000000
42591,2026-01-17 00:00:00,6988.00,6997.50,6973.50,6977.75,345435,0.0,ES,240min,2026-01-16,...,-0.692977,0.236,-0.142977,-0.906977,4.0,7.0,8.062500,6.071429,3.0,0.333333
42592,2026-01-19 04:00:00,6918.25,6935.00,6911.00,6922.00,70217,0.0,ES,240min,2026-01-19,...,-2.421659,0.236,-1.871659,-2.635659,4.0,7.0,8.062500,6.071429,3.0,0.666667


In [15]:
data_quality_report(df_final)

Data Quality Report
 - rows: 42594, cols: 188
 - top 20 NaN fractions:
swing_low              0.847021
swing_low_price        0.847021
swing_low_strength     0.847021
swing_high             0.845847
swing_high_price       0.845847
swing_high_strength    0.845847
ad_z                   0.027938
cmf                    0.027915
ad_sma_10              0.014556
ad_roc_10              0.003193
ad_roc_5               0.003029
ad_delta               0.003005
ad_dir                 0.003005
ad_rel                 0.002418
close_location         0.001503
body_pct               0.001503
wick_ratio             0.001503
volume_range           0.001503
obv_norm               0.001503
ad                     0.001503
dtype: float64

Suggestions:
 - Many columns have >50% NaNs; consider dropping or re-engineering them.
 - Drop or fix constant columns: ['openint', 'ticker', 'per', 'minute', 'minute_sin', 'minute_cos', 'is_post_break_hour', 'is_NY_open_hour']
 - `rolling_slope` may be slow; consider vect

{'rows': 42594,
 'cols': 188,
 'na_fraction': {'swing_low': 0.847020707141851,
  'swing_low_price': 0.847020707141851,
  'swing_low_strength': 0.847020707141851,
  'swing_high': 0.8458468328872611,
  'swing_high_price': 0.8458468328872611,
  'swing_high_strength': 0.8458468328872611,
  'ad_z': 0.02793820725923839,
  'cmf': 0.027914729774146593,
  'ad_sma_10': 0.01455604075691412,
  'ad_roc_10': 0.0031929379724843876,
  'ad_roc_5': 0.0030285955768418085,
  'ad_delta': 0.003005118091750012,
  'ad_dir': 0.003005118091750012,
  'ad_rel': 0.0024181809644550877,
  'close_location': 0.001502559045875006,
  'body_pct': 0.001502559045875006,
  'wick_ratio': 0.001502559045875006,
  'volume_range': 0.001502559045875006,
  'obv_norm': 0.001502559045875006,
  'ad': 0.001502559045875006,
  'clv': 0.001502559045875006,
  'ad_norm': 0.001502559045875006,
  'upper_wick_pct': 0.001502559045875006,
  'lower_wick_pct': 0.001502559045875006,
  'pct_above_ma_50': 0.0011503967694980514,
  'relative_volume': 

In [16]:
df_final.columns.tolist()

['datetime',
 'open',
 'high',
 'low',
 'close',
 'volume',
 'openint',
 'ticker',
 'per',
 'trading_date',
 'open_day',
 'high_cum',
 'low_cum',
 'volume_cum',
 'sma_20',
 'ema_20',
 'dema',
 'kama',
 'ema_fast',
 'ema_slow',
 'macd',
 'signal',
 'histogram',
 'adx',
 'rsi',
 'slowk',
 'slowd',
 'cmo',
 'williams_r',
 'cci',
 'roc_10',
 'roc_1',
 'roc_5',
 'roc_20',
 'bb_middle',
 'bb_std',
 'bb_upper',
 'bb_lower',
 'true_range',
 'range',
 'range_mean',
 'hv',
 'realized_vol_20',
 'obv',
 'obv_norm',
 'obv_roc_5',
 'obv_roc_10',
 'obv_ema_10',
 'obv_sma_10',
 'obv_z',
 'obv_dir',
 'obv_rel',
 'obv_delta',
 'vpt',
 'clv',
 'ad',
 'ad_norm',
 'ad_roc_5',
 'ad_roc_10',
 'ad_ema_10',
 'ad_sma_10',
 'ad_z',
 'ad_delta',
 'ad_dir',
 'ad_rel',
 'volume_range',
 'volume_price',
 'mfi',
 'cmf',
 'efi',
 'vwap',
 'avg_volume',
 'relative_volume',
 'volume_z',
 'volume_sum_20',
 'volume_delta',
 'pct_above_ma_20',
 'pct_above_ma_50',
 'slope_10',
 'tr_dir',
 'body',
 'wick_ratio',
 'is_doji',
